In [ ]:
"""
=============================================================================
[Code Architecture and Optimization Documentation]

1. Model Improvement (Bi-GRU):
   - Encoder: Upgrade from standard RNN to bidirectional GRU.
     - Reason: Dual GRUs process sequences in both forward and backward directions, capturing more complete context than unidirectional RNNs.
   - Decoder: Implemented a hidden state bridge (linear projection).
     - Reason: Since the encoder is bidirectional (outputs 2 hidden vectors per layer) and the decoder is unidirectional, we have to connect the forward and backward states and project them downwards to match the hidden dimensions of the decoder.

2. Object-Oriented Design (OOP):
   - Modularity: Code is refactored into specific classes:
     - Configuration: Centralized parameter management.
     - "DataEngine": Responsible for loading, cleaning, splitting, and enhancing data.
     - Trainer: Encapsulates the training loop, loss calculations, and backpropagation.
     - Evaluator: Handles inference and gauge calculations.
   - Pros: The structure is cleaner, easier to debug, and scalable.

3. Data Gain (Regularization):
   - Added 'DataEngine.augmentsequence'.
   - Mechanism: During training, the source sentence randomly undergoes:
     - Random Swap: Two words swap positions.
     - Random drop: a word is removed.
   - Benefits: This introduces noise and prevents the model from remembering specific sequences (overfitting), thereby improving generalization.

4. Evaluation Indicators (BLEU):
   - Integrated BLEU score (via NLTK) as the primary validation metric.
   - Why:Cross-entropy loss (NLLLoss) is not always associated with translation quality. BLEU measures overlap with the referenced n-gram, which is the industry standard for machine translation.

5. Hyperparameter Optimization (Optuna):
   - Automatic search is implemented through "optuna".
   - Objective: Search aims to maximize validation of BLEU.
   - Search Space: It dynamically tunes "hiddensize," "learning_rate," and "dropout."
   - Pruning: Use median pruners to block hopeless trials ahead of time, saving computational resources.

6. Strict Experimental Setup:
   - Training/Validation Segmentation: Clear 80/20 crossover ensures we make hyperparameter adjustments for unseen data.
   - Epoch-Based Training: Move from random sampling (iteration) to full dataset traversal (epoch) for consistent convergence.
   - Repeatability: The global random seed setting is set to 2025 to ensure the same results across runs.
=============================================================================
"""

import unicodedata
import string
import re
import random
import time
import math
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import optuna
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu


# =============================================================================
# 0. Hardware Check
# =============================================================================
def check_gpu():
    if not torch.cuda.is_available():
        print("\n" + "=" * 60)
        print("CRITICAL ERROR: GPU NOT DETECTED!")
        sys.exit(1)

    print(f"\n[Hardware] GPU Detected: {torch.cuda.get_device_name(0)}")
    print("[Hardware] Tensor Cores (AMP) Enabled.\n")


check_gpu()


# =============================================================================
# 1. Configuration & Utilities
# =============================================================================
class Config:
    SEED = 2025
    DEVICE = torch.device("cuda")

    # Data Settings
    LANG_SRC = "cn"
    LANG_TGT = "eng"
    MAX_LENGTH = 100
    SOS_TOKEN = 0
    EOS_TOKEN = 1
    PAD_TOKEN = 2

    BATCH_SIZE = 128
    NUM_WORKERS = 8
    PIN_MEMORY = True

    # Training Settings
    N_EPOCHS_OPT = 10
    N_EPOCHS_FULL = 200
    TEACHER_FORCING_RATIO = 0.5
    CLIP = 5.0
    PRINT_EVERY = 5
    PATIENCE = 15  # Early stopping patience

    # Optimizer Settings
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4

    # Optuna Settings
    N_TRIALS = 100


class Utils:
    @staticmethod
    def seed_everything(seed):
        random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        print(f"[System] Seed: {seed} | Device: {Config.DEVICE}")

    @staticmethod
    def as_minutes(s):
        m = math.floor(s / 60)
        s -= m * 60
        return "%dm %ds" % (m, s)

    @staticmethod
    def time_since(start, percent):
        now = time.time()
        s = now - start
        if percent == 0:
            return "0m 0s"
        es = s / percent
        rs = es - s
        return "%s (- %s)" % (Utils.as_minutes(s), Utils.as_minutes(rs))


Utils.seed_everything(Config.SEED)


# =============================================================================
# 2. Data Processing
# =============================================================================
class Vocab:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS", 2: "PAD"}
        self.n_words = 3

    def add_sentence(self, sentence):
        iterator = sentence if self.name == "cn" else sentence.split(" ")
        for word in iterator:
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1


class TranslationDataset(Dataset):
    def __init__(self, pairs, input_lang, output_lang, augment=False):
        self.pairs = pairs
        self.input_lang = input_lang
        self.output_lang = output_lang
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def augment_sequence(self, indices):
        if len(indices) <= 3:
            return indices
        new_indices = indices.copy()
        # Random Swap
        if random.random() < 0.1:
            i1, i2 = random.sample(range(len(new_indices)), 2)
            new_indices[i1], new_indices[i2] = new_indices[i2], new_indices[i1]
        # Random Drop
        if random.random() < 0.1:
            del new_indices[random.randint(0, len(new_indices) - 1)]
        return new_indices

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        if self.input_lang.name == "cn":
            input_indices = [self.input_lang.word2index[w] for w in pair[0]]
        else:
            input_indices = [self.input_lang.word2index[w] for w in pair[0].split(" ")]

        if self.augment:
            input_indices = self.augment_sequence(input_indices)

        input_indices.append(Config.EOS_TOKEN)

        if self.output_lang.name == "cn":
            target_indices = [self.output_lang.word2index[w] for w in pair[1]]
        else:
            target_indices = [
                self.output_lang.word2index[w] for w in pair[1].split(" ")
            ]
        target_indices.append(Config.EOS_TOKEN)

        return torch.tensor(input_indices, dtype=torch.long), torch.tensor(
            target_indices, dtype=torch.long
        )


class DataEngine:
    @staticmethod
    def unicode_to_ascii(s):
        return "".join(
            c
            for c in unicodedata.normalize("NFD", s)
            if unicodedata.category(c) != "Mn"
        )

    @staticmethod
    def normalize_string(s):
        s = DataEngine.unicode_to_ascii(s.lower().strip())
        s = re.sub(r"([.!?])", r" \1", s)
        s = re.sub(r"[^a-zA-Z\u4e00-\u9fa5.!?，。？]+", r" ", s)
        return s.strip()

    @staticmethod
    def prepare_data(lang1, lang2):
        filename = f"{lang1}-{lang2}.txt"
        if not os.path.exists(filename):
            print(f"[Warning] {filename} not found. Using dummy data for testing.")
            lines = ["你好\thello", "谢谢\tthank you", "再见\tgoodbye"] * 100
        else:
            lines = open(filename, encoding="utf-8").read().strip().split("\n")

        pairs = [
            [DataEngine.normalize_string(s) for s in line.split("\t")] for line in lines
        ]
        pairs = [
            p
            for p in pairs
            if len(p[0]) < Config.MAX_LENGTH
            and len(p[1].split(" ")) < Config.MAX_LENGTH
        ]

        input_lang = Vocab(lang1)
        output_lang = Vocab(lang2)

        for pair in pairs:
            input_lang.add_sentence(pair[0])
            output_lang.add_sentence(pair[1])

        print(
            f"[Data] Loaded {len(pairs)} pairs. Vocab: {input_lang.n_words}/{output_lang.n_words}"
        )
        return input_lang, output_lang, pairs

    @staticmethod
    def collate_fn(batch):
        input_batch, target_batch = zip(*batch)
        input_pad = pad_sequence(
            input_batch, padding_value=Config.PAD_TOKEN, batch_first=False
        )
        target_pad = pad_sequence(
            target_batch, padding_value=Config.PAD_TOKEN, batch_first=False
        )
        return input_pad, target_pad


# =============================================================================
# 3. Models
# =============================================================================
class EncoderBiGRU(nn.Module):
    def __init__(self, input_size, hidden_size, n_layers=1, dropout=0.0):
        super(EncoderBiGRU, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(
            hidden_size,
            hidden_size,
            n_layers,
            dropout=(0 if n_layers == 1 else dropout),
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_seq):
        embedded = self.embedding(input_seq)
        embedded = self.dropout(embedded)
        outputs, hidden = self.gru(embedded)
        return outputs, hidden


class DecoderGRU(nn.Module):
    def __init__(self, hidden_size, output_size, n_layers=1, dropout=0.0):
        super(DecoderGRU, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(
            hidden_size,
            hidden_size,
            n_layers,
            dropout=(0 if n_layers == 1 else dropout),
        )
        self.out = nn.Linear(hidden_size, output_size)
        self.bridge = nn.Linear(hidden_size * 2, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_step, hidden):
        # input_step: [Batch]
        # Embedding -> [1, Batch, Hidden]
        embedded = self.embedding(input_step).unsqueeze(0)
        embedded = self.dropout(embedded)
        output, hidden = self.gru(embedded, hidden)
        prediction = self.out(output[0])
        return F.log_softmax(prediction, dim=1), hidden

    def init_hidden(self, encoder_hidden):
        # encoder_hidden: [n_layers*2, Batch, Hidden]
        # Concat forward and backward hidden states
        hidden_fwd = encoder_hidden[-2]
        hidden_bwd = encoder_hidden[-1]
        cat_hidden = torch.cat((hidden_fwd, hidden_bwd), dim=1)
        # Project back to Decoder Hidden Size
        bridged = torch.tanh(self.bridge(cat_hidden)).unsqueeze(0)

        if self.n_layers > 1:
            bridged = bridged.repeat(self.n_layers, 1, 1)
        return bridged


# =============================================================================
# 4. Evaluator
# =============================================================================
class Evaluator:
    @staticmethod
    def evaluate(encoder, decoder, pair, input_lang, output_lang):
        """
        Performs inference on a single sentence pair.
        """
        with torch.no_grad():
            input_indices = [
                input_lang.word2index[w]
                for w in (pair[0] if input_lang.name == "cn" else pair[0].split())
            ]

            input_tensor = torch.tensor(
                input_indices,
                dtype=torch.long,
                device=Config.DEVICE,
            )
            input_tensor = input_tensor.unsqueeze(1)
            input_tensor = torch.cat(
                [
                    input_tensor,
                    torch.tensor([[Config.EOS_TOKEN]], device=Config.DEVICE),
                ],
                dim=0,
            )

            enc_out, enc_hidden = encoder(input_tensor)
            dec_hidden = decoder.init_hidden(enc_hidden)

            dec_input = torch.tensor([Config.SOS_TOKEN], device=Config.DEVICE)

            decoded_words = []
            for _ in range(Config.MAX_LENGTH):
                output, dec_hidden = decoder(dec_input, dec_hidden)
                topv, topi = output.topk(1)

                if topi.item() == Config.EOS_TOKEN:
                    decoded_words.append("<EOS>")
                    break
                decoded_words.append(output_lang.index2word[topi.item()])
                dec_input = topi.detach().view(1)

        return decoded_words

    @staticmethod
    def calculate_bleu(
        encoder, decoder, pairs, input_lang, output_lang, sample_size=500
    ):
        encoder.eval()
        decoder.eval()
        check_pairs = random.sample(pairs, min(len(pairs), sample_size))

        refs, cands = [], []
        for pair in check_pairs:
            try:
                refs.append([pair[1].split()])
                pred = Evaluator.evaluate(
                    encoder, decoder, pair, input_lang, output_lang
                )
                if pred and pred[-1] == "<EOS>":
                    pred = pred[:-1]
                cands.append(pred)
            except KeyError:
                continue

        if not cands:
            return 0
        return corpus_bleu(refs, cands, smoothing_function=SmoothingFunction().method1)

    @staticmethod
    def predict_test_file(filename, encoder, decoder, input_lang, output_lang):
        print("\n" + "=" * 60)
        print(f"[Test] Processing Test File: {filename}")

        if not os.path.exists(filename):
            print(f"[Error] {filename} not found.")
            return

        with open(filename, "r", encoding="utf-8") as f:
            lines = f.read().strip().split("\n")

        lines = [line for line in lines if line.strip()]
        total_lines = len(lines)

        num_samples = 5
        sample_indices = set(
            random.sample(range(total_lines), min(total_lines, num_samples))
        )

        output_filename = "test_results.txt"
        print(f"[Test] Total lines: {total_lines}")
        print(f"[Test] Saving results to: {output_filename}")

        encoder.eval()
        decoder.eval()

        with open(output_filename, "w", encoding="utf-8") as out_f:
            for i, line in enumerate(lines):
                src_sentence = DataEngine.normalize_string(line)

                try:
                    output_words = Evaluator.evaluate(
                        encoder, decoder, [src_sentence, ""], input_lang, output_lang
                    )

                    if output_words and output_words[-1] == "<EOS>":
                        output_sentence = " ".join(output_words[:-1])
                    else:
                        output_sentence = " ".join(output_words)

                    out_f.write(f"{line.strip()}\t{output_sentence}\n")

                    if i in sample_indices:
                        print(f"[{i}] Src: {line.strip()}")
                        print(f"    Out: {output_sentence}")
                        print("-" * 30)

                except KeyError as e:
                    err_msg = "Error: Unknown word in input"
                    out_f.write(f"{line.strip()}\t{err_msg}\n")
                    if i in sample_indices:
                        print(f"[{i}] Src: {line.strip()}")
                        print(f"    Err: OOV (Unknown Word) -> {e}")
                        print("-" * 30)

        print("[Test] Translation finished. Results saved.")
        print("=" * 60 + "\n")


# =============================================================================
# 5. Trainer
# =============================================================================
class Trainer:
    def __init__(self, encoder, decoder, lr, weight_decay=1e-5):
        self.encoder = encoder
        self.decoder = decoder

        params = list(encoder.parameters()) + list(decoder.parameters())
        self.optimizer = optim.AdamW(
            params, lr=lr, weight_decay=weight_decay, fused=True
        )

        # CHANGE: Set mode to 'min' to monitor Loss instead of BLEU
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.5, patience=5
        )

        self.criterion = nn.NLLLoss(ignore_index=Config.PAD_TOKEN)
        self.scaler = torch.amp.GradScaler("cuda")

    def train_epoch(self, loader):
        self.encoder.train()
        self.decoder.train()
        total_loss = 0

        for src, tgt in loader:
            src, tgt = (
                src.to(Config.DEVICE, non_blocking=True),
                tgt.to(Config.DEVICE, non_blocking=True),
            )
            batch_size = src.size(1)

            self.optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                enc_out, enc_hidden = self.encoder(src)
                dec_hidden = self.decoder.init_hidden(enc_hidden)

                dec_input = torch.full(
                    (batch_size,),
                    Config.SOS_TOKEN,
                    device=Config.DEVICE,
                    dtype=torch.long,
                )
                loss = 0
                use_tf = random.random() < Config.TEACHER_FORCING_RATIO
                target_len = tgt.size(0)

                # RNNs process step-by-step
                for t in range(target_len):
                    output, dec_hidden = self.decoder(dec_input, dec_hidden)
                    loss += self.criterion(output, tgt[t])
                    if use_tf:
                        dec_input = tgt[t]
                    else:
                        topi = output.argmax(1)
                        dec_input = topi.detach()

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.encoder.parameters(), Config.CLIP)
            torch.nn.utils.clip_grad_norm_(self.decoder.parameters(), Config.CLIP)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            total_loss += loss.item() / target_len

        return total_loss / len(loader)

    def fit(
        self,
        train_pairs,
        val_pairs,
        epochs,
        augment=False,
        input_lang=None,
        output_lang=None,
        trial=None,
        patience=None,
        save_path=None,
    ):
        ds = TranslationDataset(train_pairs, input_lang, output_lang, augment=augment)
        loader = DataLoader(
            ds,
            batch_size=Config.BATCH_SIZE,
            shuffle=True,
            num_workers=Config.NUM_WORKERS,
            pin_memory=Config.PIN_MEMORY,
            collate_fn=DataEngine.collate_fn,
        )

        start = time.time()
        history = {"loss": [], "bleu": []}

        best_bleu = -float("inf")
        no_improve_epochs = 0

        for epoch in range(1, epochs + 1):
            loss = self.train_epoch(loader)

            bleu = Evaluator.calculate_bleu(
                self.encoder, self.decoder, val_pairs, input_lang, output_lang
            )

            self.scheduler.step(loss)

            history["loss"].append(loss)
            history["bleu"].append(bleu)

            if epoch % Config.PRINT_EVERY == 0 or epoch == epochs:
                current_lr = self.optimizer.param_groups[0]["lr"]
                print(
                    f"{Utils.time_since(start, epoch / epochs)} (Ep {epoch}) "
                    f"Loss: {loss:.4f} | BLEU: {bleu:.4f} | LR: {current_lr:.6f}"
                )

            # Optuna Pruning
            if trial:
                trial.report(bleu, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

            # Best Model Saving & Early Stopping
            if save_path:
                if bleu > best_bleu:
                    best_bleu = bleu
                    no_improve_epochs = 0
                    torch.save(
                        {
                            "encoder": self.encoder.state_dict(),
                            "decoder": self.decoder.state_dict(),
                            "optimizer": self.optimizer.state_dict(),
                            "epoch": epoch,
                            "bleu": bleu,
                        },
                        save_path,
                    )
                    if epoch > 1:
                        print(f"  >>> [Model Saved] New Best BLEU: {bleu:.4f}")
                else:
                    no_improve_epochs += 1

                if patience and no_improve_epochs >= patience:
                    print(
                        f"\n[Early Stopping] Triggered! No improvement for {patience} epochs."
                    )
                    print(
                        f"  >>> Stopping training at Epoch {epoch}. Best BLEU: {best_bleu:.4f}"
                    )
                    break

        # Load best model
        if save_path and os.path.exists(save_path):
            print(f"\n[Model Recovery] Loading best weights from {save_path}...")
            checkpoint = torch.load(save_path)
            self.encoder.load_state_dict(checkpoint["encoder"])
            self.decoder.load_state_dict(checkpoint["decoder"])
            print(
                f"  >>> Loaded successfully. Best BLEU (Validation): {checkpoint['bleu']:.4f}"
            )

        return history


# =============================================================================
# 6. Main
# =============================================================================
def objective(trial, input_lang, output_lang, train_pairs, val_pairs):
    hidden_size = trial.suggest_categorical("hidden_size", [128, 256, 512, 1024, 2048])
    n_layers = trial.suggest_int("n_layers", 1, 4)
    lr = trial.suggest_float("lr", 5e-5, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.6)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    encoder = EncoderBiGRU(input_lang.n_words, hidden_size, n_layers, dropout).to(
        Config.DEVICE
    )
    decoder = DecoderGRU(hidden_size, output_lang.n_words, n_layers, dropout).to(
        Config.DEVICE
    )

    trainer = Trainer(encoder, decoder, lr=lr, weight_decay=weight_decay)
    hist = trainer.fit(
        train_pairs,
        val_pairs,
        epochs=Config.N_EPOCHS_OPT,
        augment=False,
        input_lang=input_lang,
        output_lang=output_lang,
        trial=trial,
    )
    return hist["bleu"][-1]


def main():
    # 1. Prepare Data
    input_lang, output_lang, pairs = DataEngine.prepare_data(
        Config.LANG_SRC, Config.LANG_TGT
    )

    random.shuffle(pairs)
    split_at = int(len(pairs) * 0.8)
    train_pairs, val_pairs = pairs[:split_at], pairs[split_at:]

    # 2. Optuna Search
    print(f"\n[Optuna] Starting Massive Search (Trials: {Config.N_TRIALS})...")
    study = optuna.create_study(direction="maximize")
    study.optimize(
        lambda t: objective(t, input_lang, output_lang, train_pairs, val_pairs),
        n_trials=Config.N_TRIALS,
    )

    print(f"[Optuna] Best Params: {study.best_params}")

    best = study.best_params
    n_layers = best.get("n_layers", 1)
    weight_decay = best.get("weight_decay", Config.WEIGHT_DECAY)

    # 3. Final Training with Best Params
    encoder = EncoderBiGRU(
        input_lang.n_words, best["hidden_size"], n_layers, best["dropout"]
    ).to(Config.DEVICE)
    decoder = DecoderGRU(
        best["hidden_size"], output_lang.n_words, n_layers, best["dropout"]
    ).to(Config.DEVICE)

    trainer = Trainer(encoder, decoder, lr=best["lr"], weight_decay=weight_decay)

    print(
        f"\n[Main] Final Training (Epochs: {Config.N_EPOCHS_FULL}) with Early Stopping..."
    )

    # Train with Early Stopping and Model Saving enabled
    hist = trainer.fit(
        train_pairs,
        val_pairs,
        epochs=Config.N_EPOCHS_FULL,
        augment=True,
        input_lang=input_lang,
        output_lang=output_lang,
        patience=Config.PATIENCE,
        save_path="best_model.pth",
    )

    # 4. Save Training Plot
    plt.figure(figsize=(10, 4), dpi=300)
    plt.subplot(1, 2, 1)
    plt.plot(hist["loss"], label="Loss")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(hist["bleu"], label="BLEU", c="orange")
    plt.legend()
    plt.savefig("result_final.png")
    print("[Main] Results saved.")

    # 5. Validation Samples
    print("\n[Validation] Check Samples (using Best Saved Model):")
    for _ in range(3):
        p = random.choice(val_pairs)
        try:
            res = Evaluator.evaluate(encoder, decoder, p, input_lang, output_lang)
            print(f"SRC: {p[0]}\nTGT: {p[1]}\nOUT: {' '.join(res)}\n---")
        except KeyError:
            pass

    # 6. Final Test on 'test.txt'
    Evaluator.predict_test_file("test.txt", encoder, decoder, input_lang, output_lang)


if __name__ == "__main__":
    main()